In [ ]:
print("hello world")

In [ ]:
from google.colab import ai
response = ai.generate_text("What is the capital of France?")
print(response)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import xml.etree.ElementTree as ET
import os
import concurrent.futures
import random

# 🚀 Google Drive Mount (for saving file)
from google.colab import drive
drive.mount('/content/drive')

# 📌 File Paths
SITEMAP_URL = "https://digestlibrary.com/post-sitemap.xml"
SAVE_PATH_XLSX = "/content/drive/My Drive/Blogger_Novels.xlsx"
FAILED_LINKS_FILE = "/content/drive/My Drive/failed_links.txt"
PROGRESS_FILE = "/content/drive/My Drive/progress.txt"

# 🛡 Headers & Session
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}
session = requests.Session()
session.headers.update(HEADERS)

# 🔄 Resume Last Progress
last_index = 0
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r") as f:
        last_index = int(f.read().strip())

# 📥 Step 1: Extract all post URLs from sitemap
response = session.get(SITEMAP_URL)
if response.status_code == 200:
    root = ET.fromstring(response.content)
    post_urls = [elem.text for elem in root.findall(".//{http://www.sitemaps.org/schemas/sitemap/0.9}loc")]
    print(f"✅ Found {len(post_urls)} posts to scrape.")
else:
    print("❌ Failed to fetch sitemap.")
    exit()

# 🔍 Step 2: Scrape Each Novel Post with Auto-Retry
def scrape_post(post_url):
    """ Extracts title and ALL download links from a Blogger post """
    retries = 5
    for attempt in range(retries):
        try:
            response = session.get(post_url, timeout=30)
            if response.status_code != 200:
                print(f"⚠ Retrying ({attempt+1}/{retries}) for: {post_url}")
                time.sleep(5)
                continue

            soup = BeautifulSoup(response.text, "html.parser")

            # Extract Title
            title = soup.find("h1")
            if not title:
                title = soup.find("h2")  # Sometimes titles are in h2

            title = title.text.strip() if title else "No Title Found"

            # Extract Download Links
            all_links = [a["href"] for a in soup.find_all("a", href=True)]
            google_drive_links = [link for link in all_links if "drive.google" in link]
            mediafire_links = [link for link in all_links if "mediafire" in link]

            return {
                "Title": title,
                "Google Drive Links": ", ".join(google_drive_links) if google_drive_links else "No Google Drive Link",
                "Mediafire Links": ", ".join(mediafire_links) if mediafire_links else "No Mediafire Link"
            }
        except requests.exceptions.RequestException as e:
            print(f"⚠ Attempt {attempt+1} failed for {post_url}: {e}")
        time.sleep(5)

    # ❌ Save failed link
    with open(FAILED_LINKS_FILE, "a") as f:
        f.write(post_url + "\n")

    return None  # Return None if all retries fail

# 📌 Step 3: Scrape Each Post & Save Data in Batches
novels_data = []
BATCH_SIZE = 100

with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:  # Max 3 workers to avoid bans
    future_to_url = {executor.submit(scrape_post, post_urls[idx]): idx for idx in range(last_index, len(post_urls))}

    for future in concurrent.futures.as_completed(future_to_url):
        idx = future_to_url[future]
        result = future.result()
        if result:
            novels_data.append(result)

        # ✅ Save every batch of 100 posts
        if len(novels_data) >= BATCH_SIZE:
            df = pd.DataFrame(novels_data)

            # 🔹 Append to existing Excel
            if os.path.exists(SAVE_PATH_XLSX):
                existing_df = pd.read_excel(SAVE_PATH_XLSX, engine='openpyxl')
                df = pd.concat([existing_df, df], ignore_index=True)

            df.to_excel(SAVE_PATH_XLSX, index=False, engine='openpyxl')

            novels_data = []  # Clear batch
            with open(PROGRESS_FILE, "w") as f:
                f.write(str(idx))

        time.sleep(random.uniform(1, 2))  # Random delay

# 📥 Step 4: Final Save
if novels_data:
    df = pd.DataFrame(novels_data)

    # 🔹 Append to Excel
    if os.path.exists(SAVE_PATH_XLSX):
        existing_df = pd.read_excel(SAVE_PATH_XLSX, engine='openpyxl')
        df = pd.concat([existing_df, df], ignore_index=True)

    df.to_excel(SAVE_PATH_XLSX, index=False, engine='openpyxl')

# ✅ Delete progress file after successful completion
if os.path.exists(PROGRESS_FILE):
    os.remove(PROGRESS_FILE)

print(f"✅ Scraping complete! Data saved in '{SAVE_PATH_XLSX}' (Google Drive)")
